# p142 Linked List Cycle II 解析笔记

- LeetCode: https://leetcode.cn/problems/linked-list-cycle-ii/
- 原始代码位置：`solutions-java/leetcode/p142_linked_list_cycle_ii/LinkedListCycleII.java`
- 题型：链表 / 快慢指针
- 难度：Medium

这道题要求我们判断链表是否有环；如果有，不只是返回 `True`，而是要进一步找出第一次进入环的那个节点。

这里的讲解和 Python 实现都直接参考 `leetcode-java` 目录中的原代码思路来整理。


## 原代码思路整理

Java 原代码采用 Floyd 快慢指针的两阶段写法，而且初始化方式也很有特点：

1. 先排除长度不足 3 的链表，因为这类链表无法安全完成原代码的初始化。
2. `slow = head.next`，`fast = head.next.next`，避免一开始就让两个指针都停在 `head`。
3. 第一阶段先判断是否有环：`slow` 走一步，`fast` 走两步；如果 `fast` 碰到 `null`，说明无环。
4. 第二阶段寻找入环点：把 `fast` 放回 `head`，让它和停在相遇点的 `slow` 同步每次走一步。
5. 两个指针第二次相遇的位置，就是入环点。

下面这版 Python 代码会尽量保持和 Java 原实现一致，而不是另写成另一种常见模板。


## 为什么第二次相遇就是入环点

这部分直接对应 Java 原注释里的证明思路。

设：

- 从头节点到入环点的距离是 `a`
- 从入环点到第一次相遇点的距离是 `b`
- 从第一次相遇点再走到入环点的距离是 `c`
- 环长就是 `b + c`

当快慢指针第一次相遇时：

- 慢指针总路程是 `a + b`
- 快指针总路程是 `a + b + k * (b + c)`

又因为快指针速度是慢指针的 2 倍，所以有：

```text
2 * (a + b) = a + b + k * (b + c)
=> a + b = k * (b + c)
=> a = c + (k - 1) * (b + c)
```

这意味着：

- 从 `head` 走到入环点要走 `a`
- 从第一次相遇点继续往前走到入环点，等价于先走 `c`，再绕若干整圈

所以一个指针从 `head` 出发，另一个指针从相遇点出发，并且两者都每次走一步时，它们一定会在入环点相遇。


In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(eq=False)
class ListNode:
    val: int
    next: ListNode | None = None


class Solution:
    def detectCycle(self, head: ListNode | None) -> ListNode | None:
        # 这一版初始化方式刻意对齐 leetcode-java 原代码。
        if head is None or head.next is None or head.next.next is None:
            return None

        slow = head.next
        fast = head.next.next

        # 第一阶段：判断是否有环。
        while slow is not fast:
            if fast.next is None or fast.next.next is None:
                return None
            slow = slow.next
            fast = fast.next.next

        # 第二阶段：从头节点和相遇点同步出发，寻找入环点。
        fast = head
        while slow is not fast:
            slow = slow.next
            fast = fast.next

        return slow


## 代码细节对应 Java 原实现

### 1. 为什么不是 `slow = head`、`fast = head`

很多题解会这么写，但原仓库 Java 代码没有这样做，而是先让快慢指针分别走 1 步和 2 步。这样 `while (slow != fast)` 可以直接进入真正的追及过程。

### 2. 为什么要先判断 `head.next.next`

因为原代码的初始化依赖 `head.next.next`。如果链表长度不足 3，就没法安全地把 `fast` 放到第二个后继节点。

### 3. 为什么第二阶段把 `fast` 放回 `head`

第一次相遇只能证明存在环，不能直接得到入环点。必须把一个指针放回头节点，才能利用上面的距离关系找到真正答案。


## 复杂度

- 时间复杂度：`O(n)`
- 空间复杂度：`O(1)`

整个过程只用了有限几个指针变量，没有使用哈希表。


## 易错点

1. 只判断是否有环，却忘了题目要求返回入环节点。
2. 第一阶段相遇后直接返回 `slow`，这通常只是相遇点，不一定是入环点。
3. 没处理短链表，导致 `head.next.next` 空指针错误。
4. 第二阶段忘了把其中一个指针放回 `head`。


In [ ]:
def build_cycle_list(values: list[int], pos: int) -> ListNode | None:
    if not values:
        return None

    nodes = [ListNode(v) for v in values]
    for i in range(len(nodes) - 1):
        nodes[i].next = nodes[i + 1]

    if pos != -1:
        nodes[-1].next = nodes[pos]

    return nodes[0]


head1 = build_cycle_list([3, 2, 0, -4], 1)
entry1 = Solution().detectCycle(head1)
print('示例一入环点:', entry1.val if entry1 else None)

head2 = build_cycle_list([1, 2], 0)
entry2 = Solution().detectCycle(head2)
print('示例二入环点:', entry2.val if entry2 else None)

head3 = build_cycle_list([1], -1)
entry3 = Solution().detectCycle(head3)
print('无环情况:', entry3)


## 一句话复盘

> 这题最重要的不是记住快慢指针能判环，而是理解第一次相遇后，为什么把一个指针放回头节点再同步前进就能找到入环点。
